<a href="https://colab.research.google.com/github/zbovaird/UHG-Library/blob/main/colab/uhg_cic_intrusion_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UHG Intrusion Detection on CIC Network Flow Data

## Purpose

Intrusion detection on CIC network flow data using **Universal Hyperbolic Geometry (UHG)**. UHG embeds data in hyperbolic space where hierarchical structure (e.g., attack vs. benign) is naturally captured.

## Methodology

- **Part A (Unsupervised)**: Build kNN graph, train ProjectiveGraphSAGE for UHG embeddings, cluster with DBSCAN, score anomalies by centroid/neighbor quadrance. No labels used for training; ideal for discovering unknown attacks.

- **Part B (Supervised)**: Same graph + ProjectiveGraphSAGE, but train with CrossEntropyLoss for multi-class classification. Use when labeled data is available and you need discrete attack-type predictions.

## Data

CIC dataset at `/content/drive/MyDrive/CIC_data.csv`. We sample 10% for faster Colab run; full data can be used by adjusting `frac`.

## Outputs

- **(A)** Anomaly scores, top-k rankings, exported detector
- **(B)** Classifier accuracy, saved model weights

In [1]:
# Install UHG 0.3.7 from GitHub. Use --force-reinstall --no-cache-dir to avoid stale cache.
!pip install --force-reinstall --no-cache-dir git+https://github.com/zbovaird/UHG-Library.git

import os
import torch
import uhg

# Mount Google Drive to access CIC_data.csv
from google.colab import drive
drive.mount('/content/drive')

# Paths - same as original notebook
FILE_PATH = '/content/drive/MyDrive/CIC_data.csv'
MODEL_SAVE_PATH = '/content/drive/MyDrive/uhg_ids_model.pth'
DETECTOR_SAVE_PATH = '/content/drive/MyDrive/uhg_ids_detector.pt'
RESULTS_PATH = '/content/drive/MyDrive/uhg_ids_results'
os.makedirs(RESULTS_PATH, exist_ok=True)

# Device configuration - prioritize GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU - training may be slower")

# Check if UHG 0.3.7+ unsupervised pipeline is available
HAS_ANOMALY = getattr(uhg, 'UHGUnsupervisedAnomalyDetector', None) is not None
print(f"UHG version: {uhg.__version__}")
print(f"Part A (unsupervised) available: {HAS_ANOMALY}")
if not HAS_ANOMALY and hasattr(uhg, '_ANOMALY_IMPORT_ERROR'):
    print("Import error:", uhg._ANOMALY_IMPORT_ERROR)

  Cloning https://github.com/zbovaird/UHG-Library.git to /tmp/pip-req-build-ypnrcj23
  Running command git clone --filter=blob:none --quiet https://github.com/zbovaird/UHG-Library.git /tmp/pip-req-build-ypnrcj23
  Resolved https://github.com/zbovaird/UHG-Library.git to commit d085aedbc6d8e4dfe54749d374d81df2ed525ce6
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using GPU: NVIDIA L4
UHG version: 0.3.7
Part A (unsupervised) available: False


In [2]:
# Load CIC data, apply schema utils for preprocessing, sample for runtime.
# Uses uhg.utils.schema (detect_label_column, enforce_numeric) when available.

import pandas as pd
import numpy as np

print(f"Loading data from: {FILE_PATH}")
data = pd.read_csv(FILE_PATH, low_memory=False)
data.columns = data.columns.str.strip()
if 'Label' in data.columns:
    data['Label'] = data['Label'].str.strip()

# Display label distribution
unique_labels = data['Label'].unique() if 'Label' in data.columns else []
print(f"\nUnique labels: {len(unique_labels)}")
if 'Label' in data.columns:
    print(data['Label'].value_counts())

# Sample 10% for faster Colab run
data_sampled = data.sample(frac=0.10, random_state=42)

# Preprocessing: use uhg schema utils if available, else manual
try:
    from uhg.utils.schema import detect_label_column, enforce_numeric
    label_col = detect_label_column(data_sampled)
    feature_cols = [c for c in data_sampled.columns if c != label_col]
    df_features = data_sampled[feature_cols].copy()
    df_features = enforce_numeric(df_features, fill='mean', replace_inf=True)
    labels_series = data_sampled[label_col] if label_col else None
except ImportError:
    df_features = data_sampled.drop(columns=['Label'], errors='ignore').apply(
        pd.to_numeric, errors='coerce'
    )
    df_features = df_features.replace([np.inf, -np.inf], np.nan)
    df_features = df_features.fillna(df_features.mean()).fillna(0)
    labels_series = data_sampled['Label'] if 'Label' in data_sampled.columns else None

print(f"\nSampled shape: {df_features.shape}")
df_sampled = df_features.copy()
if labels_series is not None:
    df_sampled['Label'] = labels_series.values

Loading data from: /content/drive/MyDrive/CIC_data.csv

Unique labels: 15
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64

Sampled shape: (283074, 77)


In [3]:
# Part A: Unsupervised anomaly detection. Requires UHG 0.3.7+.
# fit_from_dataframe auto-detects Label column, clusters, scores, exports detector.

if HAS_ANOMALY:
    Detector = uhg.UHGUnsupervisedAnomalyDetector
    detector = Detector(hidden=64, embedding_dim=32)
    detector.fit_from_dataframe(df_sampled, epochs=50, k=5, seed=42)

    # Optional: suggest eps via auto_eps_kdist before clustering
    try:
        from uhg.cluster.dbscan import auto_eps_kdist
        eps_suggested = auto_eps_kdist(detector.embeddings.numpy(), k=4)
        print(f"Suggested eps (k-dist): {eps_suggested:.4f}")
    except Exception:
        pass

    detector.cluster(eps=0.5, min_samples=3)
    scores = detector.score(method='centroid_quadrance')
    summary = detector.summarize(topk=20)
    print("Timings:", summary.get('timings', {}))
    print("Top entities:", summary.get('top_entities', [])[:10])
    if summary.get('cluster_metrics'):
        print("Cluster metrics:", summary['cluster_metrics'])

    detector.export(DETECTOR_SAVE_PATH)
    print(f"Detector exported to {DETECTOR_SAVE_PATH}")
else:
    print("Part A requires UHG 0.3.7+. Install from GitHub after it's pushed:")
    print("  !pip install git+https://github.com/zbovaird/UHG-Library.git")

Part A requires UHG 0.3.7+. Install from GitHub after it's pushed:
  !pip install git+https://github.com/zbovaird/UHG-Library.git


In [4]:
# Part A evaluation: compare top-k anomalies to ground-truth labels (when available).
# Not used for training - only for post-hoc evaluation.

if HAS_ANOMALY and 'Label' in df_sampled.columns:
    try:
        scores = detector.score(method='centroid_quadrance')
        top_indices = torch.topk(scores, min(100, len(scores))).indices.numpy()
        labels_arr = df_sampled['Label'].values
        benign_key = 'BENIGN'
        attack_count = sum(1 for i in top_indices if labels_arr[i] != benign_key)
        print(f"Of top-{len(top_indices)} anomalies, {attack_count} were attacks (non-BENIGN)")
    except NameError:
        print("Run Part A cell first to define detector")

In [5]:
# Part B: Supervised multi-class classification using ProjectiveGraphSAGE.
# Uses build_knn_graph, EarlyStopping from UHG 0.3.7. Requires labels.

if 'Label' not in df_sampled.columns:
    print("No Label column - skipping Part B")
else:
    try:
        from sklearn.preprocessing import StandardScaler
        from uhg.graph.build import build_knn_graph
        from uhg.nn.models.sage import ProjectiveGraphSAGE
        from uhg.nn.early_stopping import EarlyStopping

        # Prepare features and labels
        scaler = StandardScaler()
        X = scaler.fit_transform(df_features.values.astype(np.float32))
        label_mapping = {lb: i for i, lb in enumerate(unique_labels)}
        y = np.array([label_mapping.get(lb, 0) for lb in labels_series])

        # Build kNN graph (library function replaces sklearn kneighbors_graph)
        edge_index = build_knn_graph(X, k=5)
        x_t = torch.tensor(X, dtype=torch.float32).to(device)
        edge_index = edge_index.to(device)
        y_t = torch.tensor(y, dtype=torch.long).to(device)

        # Train/val/test split (70/15/15)
        n = len(x_t)
        perm = torch.randperm(n, device=device)
        train_end = int(0.7 * n)
        val_end = train_end + int(0.15 * n)
        train_mask = torch.zeros(n, dtype=torch.bool, device=device)
        val_mask = torch.zeros(n, dtype=torch.bool, device=device)
        test_mask = torch.zeros(n, dtype=torch.bool, device=device)
        train_mask[perm[:train_end]] = True
        val_mask[perm[train_end:val_end]] = True
        test_mask[perm[val_end:]] = True

        # Model: ProjectiveGraphSAGE (adds homogeneous coord internally)
        in_ch = x_t.size(1)
        num_classes = len(label_mapping)
        model = ProjectiveGraphSAGE(
            in_channels=in_ch,
            hidden_channels=128,
            out_channels=num_classes,
            num_layers=2,
            dropout=0.2
        ).to(device)

        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
        criterion = torch.nn.CrossEntropyLoss()
        early_stop = EarlyStopping(min_delta=0.01, patience=20, mode='max')

        # Training loop
        for epoch in range(1, 301):
            model.train()
            optimizer.zero_grad()
            out = model(x_t, edge_index)
            loss = criterion(out[train_mask], y_t[train_mask])
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                pred = model(x_t, edge_index).argmax(dim=1)
                val_acc = (pred[val_mask] == y_t[val_mask]).float().mean().item()
            if early_stop(val_acc, model):
                print(f"Early stopping at epoch {epoch}")
                break
            if epoch % 20 == 0:
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}, Val Acc: {val_acc:.4f}")

        early_stop.restore_best(model)
        model.to(device)
        test_acc = (model(x_t, edge_index).argmax(dim=1)[test_mask] == y_t[test_mask]).float().mean().item()
        print(f"Final test accuracy: {test_acc:.4f}")
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"Model saved to {MODEL_SAVE_PATH}")
    except ImportError as e:
        print(f"Part B requires UHG 0.3.7 (build_knn_graph, ProjectiveGraphSAGE, EarlyStopping): {e}")

Part B requires UHG 0.3.7 (build_knn_graph, ProjectiveGraphSAGE, EarlyStopping): No module named 'uhg.nn.models.hgnn'


## Summary

- **Part A (Unsupervised)**: Use when you want to discover anomalies without labels. Outputs anomaly scores, top-k rankings, and an exported detector. Requires UHG 0.3.7+.

- **Part B (Supervised)**: Use when you have labeled data and need attack-type classification. Outputs test accuracy and saved model weights. Requires UHG 0.3.7+ for library components.

Both workflows use the same CIC data path and save outputs to Google Drive.